In [ ]:
# DGHK23，嵌入不等式hint， <v,s> <= l
cd framework

In [ ]:
load("../framework/LWE.sage")
import numpy as np
import random

In [ ]:
n = 80
m = n
q = 1201
D_s = build_centered_binomial_law(10)
D_e = D_s
lwe_instance = LWE(n, q, m, D_e, D_s)
ebdd = lwe_instance.embed_into_EBDD()

In [ ]:
# 使用拒绝采样生成[-3*sigma,3*sigma]之间满足高斯分布的随机数
def sample_discrete_gaussian(sigma):
    while True:
        x = random.randint(int(-3 * sigma), int(3 * sigma)) 
        acceptance_prob = math.exp(-x**2 / (2 * sigma**2)) 
        if random.uniform(0, 1) < acceptance_prob: 
            return x
        
# 生成不等式hint，<v,s>=l-noisy，<v,s><l
def generate_se_eta_sigma_ineq_hint(m, n, q, sigma, k):
  V = []
  L = []

  for i in range(k):
    D_e = {-3: 1/64, -2: 6/64, -1: 15/64, 0: 20/64, 1: 15/64, 2: 6/64, 3: 1/64}
    values, probabilities = zip(*D_e.items())
    v = np.array(np.random.choice(values, size=m+n, p=probabilities))
    v_vec = vec(v)
    noisy = sample_discrete_gaussian(sigma)+ 3 * sigma
    # print(noisy)
    l = ebdd.leak(v_vec) + noisy
    V.append(v)
    L.append(l)
  print("L",L)
  return V,L

In [ ]:
sigma = 2
Num = []
Beta_est = []
Beta_pra = []
num_hint = 1001
V, L = generate_se_eta_sigma_ineq_hint(m, n, q, sigma, num_hint)
for j in range(num_hint):
    if j % 50 == 0:
        Num.append(j)
        beta_est, delta = ebdd.estimate_attack()
        Beta_est.append(beta_est)

        secret = ebdd.attack()
        beta_pra = secret[0]
        Beta_pra.append(beta_pra)
    print("the ",j+1,"-th secret error ineq hint")
    _ = ebdd.integrate_ineq_hint(vec(V[j]), L[j])
print(Num)
print("Beta_est",Beta_est)
print("Beta_pra",Beta_pra)

In [ ]:
output_dir = "/root/ShaoMingYao/Lattice_Reduction/DGHK23/Geometric-LWE-Estimator-master/framework/Prac_ineq_hint/LWE_80_10_3_2/T9/"

A_file_path = os.path.join(output_dir, "A.txt")
with open(A_file_path, 'w') as f:
    _ = f.write('[')
    for i in range(m):
        row = list(lwe_instance.A[i])
        _ = f.write(f'{row}')
        if i < m - 1:
            _ = f.write(',\n')
    _ = f.write(']')

b_file_path = os.path.join(output_dir, "b.txt")
with open(b_file_path, 'w') as f:
    _ = f.write('[')
    for i in range(m):
        _ = f.write(f'{lwe_instance.b[0][i]}')
        if i < m - 1:
            _ = f.write(', ')
    _ = f.write(']')

# 存储私钥es.txt

es_str = " ".join(map(str, es))
es_file_path = os.path.join(output_dir, "es.txt")
with open(es_file_path, "w") as f:
    _ = f.write(es_str) # _ = 是为了显式处理返回值，否则f.write(es_str) 方法的返回值是写入文件的字符数
print(f"Secret es has been saved to {es_file_path}")

# 存储系数v.txt

v_file_path = os.path.join(output_dir, "V.txt")
with open(v_file_path, "w") as vf:
    for row in V:
        _ = vf.write(" ".join(map(str, row)) + "\n")
print(f"Matrix V has been saved to {v_file_path}")

# 存储系数l.txt
l_file_path = os.path.join(output_dir, "l.txt")
with open(l_file_path, "w") as lf:
    for value in L:
        _ = lf.write(str(value) + "\n")
print(f"Vector L has been saved to {l_file_path}")


In [ ]:
L_ori = []

for j in range(num_hint):
    es = lwe_instance.e_vec.list()+lwe_instance.s.list()
    noisy = sample_discrete_gaussian(sigma)+ 3 * sigma
    l = np.dot(vec(V[j]),es) + noisy
    L_ori.append(int(l))

l_file_path = os.path.join(output_dir, "l_ori.txt")
with open(l_file_path, "w") as lf:
    for value in L_ori:
        _ = lf.write(str(value) + "\n")
print(f"Vector L has been saved to {l_file_path}")